# Ornith-1.5-35B-A3B (CRACK, uncensored) — OpenAI-compatible API on Kaggle **P100**

**Model:** `dealignai/Ornith-1.5-35B-A3B-UNCENSORED-GGUF` (Q2_K, 13.25 GB) · **Server:** llama.cpp `llama-server` · **Tunnel:** Cloudflare quick tunnel

MoE sibling of Qwen3.8-27B: hybrid GatedDeltaNet + attention, **256 experts / 8 active + 1 shared** (~3B of 35B params active per token), MTP head, native **262,144-token context**, vision (mmproj *not* loaded here — text-only, keeps VRAM for context).

> Fallback for when **TPU v5e-8** is unavailable. Expected speed on P100: **~40-90 tok/s** decode (only ~0.9 GB of weights read per token at Q2_K — MoE beats the 27B dense's 15-22 tok/s).

In [ ]:
import subprocess, sys
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,compute_cap",
                    "--format=csv"], capture_output=True, text=True)
print(r.stdout or r.stderr)
out = r.stdout or ""
if "P100" not in out:
    print("WARNING: expected GPU P100. Session options -> Accelerator -> GPU P100, then rerun.",
          file=sys.stderr)
else:
    print("OK: P100 (compute capability 6.0) detected - build will target sm_60")

In [ ]:
%%bash
set -e
mkdir -p /kaggle/tmp/models /kaggle/tmp/logs
cat > /kaggle/tmp/config.env <<EOF
MODEL_REPO=dealignai/Ornith-1.5-35B-A3B-UNCENSORED-GGUF
MODEL_FILE=Ornith-1.5-35B-A3B-CRACK-Q2_K.gguf
ALIAS=Ornith-1.5-35B-A3B-CRACK
API_KEY=1e0afcc97b0ba77076ef35a63664d578
EOF
# cloudflared binary (quick tunnel, no account needed)
if [ ! -x /kaggle/tmp/cloudflared ]; then
  curl -sL -o /kaggle/tmp/cloudflared \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  chmod +x /kaggle/tmp/cloudflared
fi
/kaggle/tmp/cloudflared --version

In [ ]:
%%bash
set -e
source /kaggle/tmp/config.env
if [ -x /kaggle/tmp/llama.cpp/build/bin/llama-server ]; then
  echo "llama.cpp already built - skipping"
else
  rm -rf /kaggle/tmp/llama.cpp
  git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/tmp/llama.cpp
  # Kaggle's image has no libcuda.so* anywhere (not even toolkit stubs), which
  # breaks cmake's CUDA::cuda_driver imported target. GGML_CUDA_NO_VMM removes
  # that link: VMM (cuMemCreate) only benefits modern GPUs; nothing lost on Pascal.
  # P100 = compute capability 6.0 (sm_60). Kaggle's CUDA 12.x toolkit still supports Pascal.
  cmake -S /kaggle/tmp/llama.cpp -B /kaggle/tmp/llama.cpp/build \
    -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=60 \
    -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF
  cmake --build /kaggle/tmp/llama.cpp/build --config Release -j4
fi
/kaggle/tmp/llama.cpp/build/bin/llama-server --version | head -1

In [ ]:
%%bash
set -e
source /kaggle/tmp/config.env
# NB: the old `huggingface-cli download` syntax is gone in current huggingface_hub
# (replaced by the `hf` CLI). Use the stable Python API instead.
python3 - <<'PY'
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
from huggingface_hub import snapshot_download
repo = "dealignai/Ornith-1.5-35B-A3B-UNCENSORED-GGUF"
f = "Ornith-1.5-35B-A3B-CRACK-Q2_K.gguf"
import os
if os.path.exists(f"/kaggle/tmp/models/{f}"):
    print("model already present - skipping")
else:
    snapshot_download(repo_id=repo, allow_patterns=[f], local_dir="/kaggle/tmp/models")
PY
ls -lh /kaggle/tmp/models/
df -h /kaggle/tmp | tail -1

In [ ]:
%%bash
source /kaggle/tmp/config.env
pkill -x llama-server 2>/dev/null; sleep 2
BIN=/kaggle/tmp/llama.cpp/build/bin/llama-server
UP=0
# 128k context with q4_0 KV (10 full-attn layers -> ~1.3 GB KV at 256k). The ngl/FA
# ladder backs off if VRAM allocation fails at load time.
for FLAGS in "-c 262144 -fa on --cache-type-k q4_0 --cache-type-v q4_0" \
             "-c 262144 -fa on --cache-type-k q8_0 --cache-type-v q8_0" \
             "-c 262144 -fa off"; do
  for NGL in 99 90 80 60; do
    echo "== trying: $FLAGS -ngl $NGL =="
    nohup $BIN \
        -m "/kaggle/tmp/models/$MODEL_FILE" \
        -a "$ALIAS" \
        --host 127.0.0.1 --port 8080 \
        -ngl $NGL $FLAGS \
        --cache-reuse 256 --jinja --no-webui \
        --threads 4 \
        > /kaggle/tmp/logs/server.log 2>&1 &
    PID=$!
    OK=0
    for i in $(seq 1 60); do
      sleep 2
      if curl -s http://127.0.0.1:8080/health 2>/dev/null | grep -q '"ok"'; then OK=1; break; fi
      kill -0 $PID 2>/dev/null || break
    done
    if [ "$OK" = "1" ]; then
      echo "SERVER UP with $FLAGS -ngl $NGL"
      UP=1
      break 2
    fi
    pkill -x llama-server 2>/dev/null; sleep 2
  done
done
if [ "$UP" != "1" ]; then
  echo "ALL ATTEMPTS FAILED - last log:"; tail -30 /kaggle/tmp/logs/server.log; exit 1
fi
tail -5 /kaggle/tmp/logs/server.log

In [ ]:
%%bash
source /kaggle/tmp/config.env
pkill -x cloudflared 2>/dev/null; sleep 2
URL=""
for ATTEMPT in 1 2 3; do
  nohup /kaggle/tmp/cloudflared tunnel --url http://127.0.0.1:8080 --no-autoupdate \
    > /kaggle/tmp/logs/tunnel.log 2>&1 &
  for i in $(seq 1 30); do
    sleep 2
    URL=$(grep -oE 'https://[a-z0-9-]+\.trycloudflare\.com' /kaggle/tmp/logs/tunnel.log | head -1)
    [ -n "$URL" ] && break
  done
  [ -n "$URL" ] && break
  echo "--- tunnel attempt $ATTEMPT failed; log tail: ---"
  tail -12 /kaggle/tmp/logs/tunnel.log 2>/dev/null || echo "(no tunnel.log)"
  pkill -x cloudflared 2>/dev/null; sleep 2
done
if [ -z "$URL" ]; then
  echo "TUNNEL FAILED after 3 attempts"
  exit 1
fi
echo ""
echo "======================================================================"
echo "  ENDPOINT : $URL/v1"
echo "  API KEY  : $API_KEY"
echo "  MODEL    : $ALIAS"
echo "======================================================================"
echo ""
echo "wire it into opencode locally:"
echo "  cd ~/Documents/Default\\ Project/colab-qwen && ./set-colab-url.sh $URL" 

In [ ]:
%%bash
source /kaggle/tmp/config.env
curl -s http://127.0.0.1:8080/v1/chat/completions \
  -H "Content-Type: application/json" -H "Authorization: Bearer $API_KEY" \
  -d '{"model":"'"$ALIAS"'","messages":[{"role":"user","content":"Reply with exactly: API OK"}],"max_tokens":32}' \
  | python3 -c "import json,sys; r=json.load(sys.stdin); print(r['choices'][0]['message']['content']); print('usage:', r.get('usage'))"
echo ""
echo "speed check (20 tokens):"
curl -s http://127.0.0.1:8080/v1/chat/completions \
  -H "Content-Type: application/json" -H "Authorization: Bearer $API_KEY" \
  -d '{"model":"'"$ALIAS"'","messages":[{"role":"user","content":"Count from 1 to 20"}],"max_tokens":64}' \
  | python3 -c "
import json,sys,time
t0=time.time(); r=json.load(sys.stdin)
u=r.get('usage',{})
dt=time.time()-t0
c=u.get('completion_tokens',0)
print(f'{c} tokens in {dt:.1f}s = {c/dt:.1f} tok/s (incl. network+parse)')"